# Hello Garage S3 + Spark

This notebook uploads a small CSV to Garage, reads it through S3A, aggregates it with PySpark, and writes the result back to Garage.

In [ ]:
import os
import boto3

s3 = boto3.client(
    's3',
    endpoint_url=os.environ['S3_ENDPOINT'],
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
)
with open('/home/jovyan/work/datasets/sample.csv', 'rb') as sample_file:
    s3.upload_fileobj(sample_file, 'datasets', 'sample.csv')
print('Sample dataset uploaded to Garage.')

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, sum as sum_

spark = (
    SparkSession.builder
    .appName('hello-garage-spark')
    .master('spark://spark-master:7077')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

In [ ]:
input_df = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv('s3a://datasets/sample.csv')
)
input_df.show()

In [ ]:
result_df = input_df.groupBy('category').agg(
    count('*').alias('row_count'),
    sum_('amount').alias('total_amount'),
)
result_df.show()

In [ ]:
result_df.write.mode('overwrite').parquet('s3a://student-results/hello-garage-spark/')
print('Result written to Garage.')